# Data Overview
Mario Castro

In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

d:\MasterHDXD\Escuela\Maestria\MachineLearning\IberLEF_Challenge\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = r"../Data/raw"

df_info = pd.read_csv(f"{path}/INFO_AUG.csv")
df_quest = pd.read_csv(f"{path}/QUEST_AUG.csv")
df_reflex = pd.read_csv(f"{path}/REFLEX_AUG.csv")

## Info dataframe:

In [3]:
df_info.head()

,client_utterance,clinician_utterance,behavior_code_1
0,He estado usando los ejercicios de respiración...,Estudios muestran que el 65% de los veteranos ...,GI
1,He estado usando los ejercicios de respiración...,Como ya has experimentado un avance en tu cont...,PE
2,"He intentado dejar de fumar varias veces, pero...",La nicotina en el cuerpo se reduce a la mitad ...,GI
3,"He intentado dejar de fumar varias veces, pero...",Tu fuerza en el servicio puede aplicarse ahora...,PE
4,"Sé que fumo demasiado, pero me ayuda a relajar...",El uso crónico de marihuana puede alterar la m...,GI


In [4]:
df_info.shape

(200, 3)

In [5]:
df_info['behavior_code_1'].value_counts()

behavior_code_1
GI    100
PE    100
Name: count, dtype: int64

## Quest dataframe:

In [6]:
df_quest.head()

,client_utterance,clinician_utterance,behavior_code_1
0,He estado practicando técnicas de respiración ...,¿Qué técnicas de respiración has encontrado má...,OQ
1,He estado practicando técnicas de respiración ...,¿Has notado que has tenido menos conflictos co...,CQ
2,"Hace dos semanas que dejé de fumar, y aunque a...",¿Cómo has manejado las ganas de fumar desde qu...,OQ
3,"Hace dos semanas que dejé de fumar, y aunque a...",¿Has tenido algún momento en el que hayas sent...,CQ
4,"He estado pensando en dejar de fumar, pero es ...",¿Qué estrategias has usado antes para manejar ...,OQ


In [7]:
df_quest.shape

(200, 3)

In [8]:
df_quest['behavior_code_1'].value_counts()

behavior_code_1
OQ    100
CQ    100
Name: count, dtype: int64

## Reflex dataframe:

In [9]:
df_reflex.head()

,client_utterance,clinician_utterance,behavior_code_1
0,He empezado a seguir una rutina de sueño más r...,Parece que estás encontrando el coraje para pr...,CR
1,No puedo cumplir con las visitas de mi supervi...,No puedes cumplir con las visitas de tu superv...,SR
2,He estado cumpliendo con todas las citas de mi...,Entiendo que has cumplido con todas las citas ...,SR
3,"No necesito seguir un horario para dormir, est...",Estás siendo defensivo para evitar reconocer q...,CR
4,No hago nada más que sentarme en casa todo el ...,Sientes que no haces nada más que sentarte en ...,SR


In [10]:
df_reflex['behavior_code_1'].value_counts()

behavior_code_1
SR    102
CR     98
Name: count, dtype: int64

In [11]:
df_reflex.shape

(200, 3)

## Pipeline

In [12]:
import os
os.environ["HF_HOME"] = "D:/huggingface_cache"

import torch
import pandas as pd
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

MODEL_NAME = "datificate/gpt2-small-spanish"
MAX_LEN     = 256
EPOCHS      = 5
LR          = 3e-5
BATCH_SIZE  = 4
SAMPLES     = 100
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(DEVICE)

cuda


In [13]:
def format_example(row):
    return f"{row['behavior_code_1']} [SEP] client: {row['client_utterance']} clinician: {row['clinician_utterance']}"

class ConversationDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.examples = []
        for _, row in df.iterrows():
            text     = format_example(row)
            encoding = tokenizer(
                text,
                max_length=MAX_LEN,
                truncation=True,
                padding="max_length",
                return_tensors="pt",
            )
            input_ids = encoding["input_ids"].squeeze()
            self.examples.append({
                "input_ids": input_ids,
                "labels":    input_ids.clone(),
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

In [14]:
def load_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    model.to(DEVICE)

    return model, tokenizer

model, tokenizer = load_model()
print(f"Model loaded on {DEVICE}")

d:\MasterHDXD\Escuela\Maestria\MachineLearning\IberLEF_Challenge\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MasterHDXD\.cache\huggingface\hub\models--datificate--gpt2-small-spanish. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 149/149 [00:00<00:00, 24939.20

Model loaded on cuda


In [17]:
def finetune(model, tokenizer, df):
    dataset    = ConversationDataset(df, tokenizer)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    optimizer  = AdamW(model.parameters(), lr=LR)

    model.train()
    for epoch in range(EPOCHS):
        total_loss = 0
        for batch in dataloader:
            input_ids = batch["input_ids"].to(DEVICE)
            labels    = batch["labels"].to(DEVICE)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, labels=labels)
            loss    = outputs.loss
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch + 1}/{EPOCHS} — Loss: {avg_loss:.4f}")



In [20]:
def generate_examples(model, tokenizer, classes, samples_per_class=SAMPLES):
    model.eval()
    generated_rows = []

    for class_label in classes:
        print(f"Generating for class: {class_label}")
        prompt   = f"{class_label} [SEP]"
        count    = 0
        attempts = 0

        while count < samples_per_class and attempts < samples_per_class * 3:
            attempts += 1
            input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)

            with torch.no_grad():
                output = model.generate(
                    input_ids,
                    max_new_tokens=MAX_LEN,
                    do_sample=True,
                    temperature=0.85,
                    top_p=0.92,
                    repetition_penalty=1.3,
                    pad_token_id=tokenizer.eos_token_id,
                )

            decoded = tokenizer.decode(output[0], skip_special_tokens=True)

            try:
                client_part    = decoded.split("client:")[1].split("clinician:")[0].strip()
                clinician_part = decoded.split("clinician:")[1].strip()
                if client_part and clinician_part:
                    generated_rows.append({
                        "client_utterance":    client_part,
                        "clinician_utterance": clinician_part,
                        "behavior_code_1":     class_label,
                    })
                    count += 1
            except IndexError:
                continue

        print(f"  Generated {count}/{samples_per_class} examples")

    return pd.DataFrame(generated_rows)



In [22]:
def save_and_clear(generated_df, output_path, model, tokenizer):
    # Save CSV
    generated_df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Saved {len(generated_df)} examples to {output_path}")

    # Clear GPU memory
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print("GPU memory cleared")


# Objective:
Participants must provide three distinct datasets, one for each pair of Behavior Codes (BCs):

    1. Reflection:
    Simple Reflection vs. Complex Reflection (SR, CR)
    2. Questions:
    Open Question vs. Closed Question (OQ, CQ)
    3. Information:
    Persuasion vs. Giving Information (PE, GI)


Constraints: Each CSV dataset must contain at least 100 and at most 200 labeled conversation turns (rows).

### SR vs. CR (df_reflex)

In [18]:
finetune(model, tokenizer, df_reflex)

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 1/5 — Loss: 4.1829
Epoch 2/5 — Loss: 1.2009
Epoch 3/5 — Loss: 0.6318
Epoch 4/5 — Loss: 0.5328
Epoch 5/5 — Loss: 0.4754


In [21]:
classes      = df_reflex["behavior_code_1"].unique().tolist()
generated_df = generate_examples(model, tokenizer, classes)
print(generated_df.head())

Generating for class: CR
  Generated 100/100 examples
Generating for class: SR
  Generated 100/100 examples
                                    client_utterance  \
0  He decidido dejar de fumar marihuana todos los...   
1  No necesito trabajar y estoy ocupado con mi tr...   
2        No puedo dormir ni la cama y estoy agotado.   
3  Me siento bien, no tengo que depender de los d...   
4  No sé cómo hacer con esto, no he estado muy bi...   

                                 clinician_utterance behavior_code_1  
0  Parece que estás pensando en dejar el hábito t...              CR  
1  Su enojo parece estar en constante conflicto p...              CR  
2  Parece que estás cansado de hacer ejercicio, l...              CR  
3  Parece que estás pensando en ti como un hecho ...              CR  
4  Parece que tu nerviosismo surge de la sensació...              CR  


In [23]:
save_and_clear(generated_df, "generated_dataset_reflex.csv", model, tokenizer)

Saved 200 examples to generated_dataset_reflex.csv
GPU memory cleared


### OQ vs. CQ (Questions)

In [24]:
finetune(model, tokenizer, df_quest)

Epoch 1/5 — Loss: 0.5378
Epoch 2/5 — Loss: 0.4184
Epoch 3/5 — Loss: 0.3570
Epoch 4/5 — Loss: 0.3049
Epoch 5/5 — Loss: 0.2653


In [25]:
classes      = df_quest["behavior_code_1"].unique().tolist()
generated_df = generate_examples(model, tokenizer, classes)
print(generated_df.head())

Generating for class: OQ
  Generated 100/100 examples
Generating for class: CQ
  Generated 100/100 examples
                                    client_utterance  \
0  Aunque he intentado reducir mi consumo de mari...   
1  Me siento abrumado con los exámenes, pero no s...   
2  A veces siento que necesito mi libertad condic...   
3  No puedo controlar lo que siento cuando pienso...   
4  No puedo concentrarme en mis estudios, cada dí...   

                                 clinician_utterance behavior_code_1  
0  ¿Cómo has logrado equilibrar tu adicción al az...              OQ  
1  ¿Cómo has estado manejando tu ira y qué desafí...              OQ  
2  ¿Has hablado alguna vez con alguien a lo largo...              OQ  
3  ¿Has notado una mejora significativa con los n...              OQ  
4  ¿Qué situaciones específicas te hacen sentir m...              OQ  


In [26]:
save_and_clear(generated_df, "generated_dataset_quest.csv", model, tokenizer)

Saved 200 examples to generated_dataset_quest.csv
GPU memory cleared


### PE vs. GI (Info)

In [27]:
finetune(model, tokenizer, df_info)

Epoch 1/5 — Loss: 0.7607
Epoch 2/5 — Loss: 0.6290
Epoch 3/5 — Loss: 0.5498
Epoch 4/5 — Loss: 0.4821
Epoch 5/5 — Loss: 0.4282


In [28]:
classes      = df_info["behavior_code_1"].unique().tolist()
generated_df = generate_examples(model, tokenizer, classes)
print(generated_df.head())

Generating for class: GI
  Generated 100/100 examples
Generating for class: PE
  Generated 100/100 examples
                                    client_utterance  \
0  ¡Estoy harto de que me digan qué hacer! Me sie...   
1  Me siento abrumado con todas estas restriccion...   
2  No necesito dejar de fumar, estoy bien y no te...   
3  No sé cómo manejar mi ira y mis estudios, ya n...   
4  He estado usando marihuana varias veces, pero ...   

                                 clinician_utterance behavior_code_1  
0  Estudios muestran una disminución en la frecue...              GI  
1  Si se gestionará, notarás una mejora en tu cap...              GI  
2  El consumo frecuente en el control del tabaqui...              GI  
3  El uso crónico de marihuana puede afectar la c...              GI  
4  El consumo frecuente de marihuana puede altera...              GI  


In [29]:
save_and_clear(generated_df, "generated_dataset_info.csv", model, tokenizer)

Saved 200 examples to generated_dataset_info.csv
GPU memory cleared
